In [1]:
using JuMP
using HiGHS
using Printf
using Random
using Distributions
import StatsBase: sample, Weights
using Graphs
using SimpleWeightedGraphs

**1**)

Vi starter med at bygge objektfunktionen som er baseret på Alastairs. Den ser således ud:  

\begin{equation}
    \max \sum_{c \in C} r_c y_c - C^{fuel} \sum_{p=1}^{|P|-1} B_p - C^{shift} \sum_{p \in P} \sum_{c \in C} \sum_{(i,j) \in S} \delta_{c,ij,p}
\end{equation}

where $r_c = 100$, $C^{fuel} = 1.0$, and $C^{shift} = 2.0$.


Definerer deck A, definerer unavailable slots, bygger movement rules, build_instance:  8 cargo, 4 ports, cost parameters, build_instance_varied — variable cargo count, random ports and arrival times with seed for reproducibility. 

In [2]:
nrows = 7
ncols = 10

function build_instance(rev, c_fuel, c_shift, h_val)

    unava = Set([(1,10),(2,10),(6,10),(7,10)])
    ramp  = [(3,10),(4,10),(5,10)]

    S = [(i,j) for i in 1:nrows for j in 1:ncols if !((i,j) in unava)]
    E = ramp
    S_no_exit = [s for s in S if !(s in E)]

    P = collect(1:4)
    C = collect(1:8)

    A     = Tuple{Tuple{Int,Int}, Tuple{Int,Int}}[]
    Gamma = Dict{Tuple{Tuple{Int,Int},Tuple{Int,Int}}, Vector{Tuple{Int,Int}}}()

    for (i,j) in S

        # East — one step forward only
        t = (i, j+1)
        if j+1 <= ncols && t in S
            push!(A, ((i,j), t))
            Gamma[((i,j), t)] = Tuple{Int,Int}[]
        end

        # West — one step backward only
        t = (i, j-1)
        if j-1 >= 1 && t in S
            push!(A, ((i,j), t))
            Gamma[((i,j), t)] = Tuple{Int,Int}[]
        end

        # North-East diagonal
        if i > 1 && j+1 <= ncols
            t = (i-1, j+1)
            if t in S
                push!(A, ((i,j), t))
                clearance = Tuple{Int,Int}[]
                (i-1, j)   in S && push!(clearance, (i-1, j))
                (i,   j+1) in S && push!(clearance, (i,   j+1))
                Gamma[((i,j), t)] = clearance
            end
        end

        # South-East diagonal
        if i < nrows && j+1 <= ncols
            t = (i+1, j+1)
            if t in S
                push!(A, ((i,j), t))
                clearance = Tuple{Int,Int}[]
                (i+1, j)   in S && push!(clearance, (i+1, j))
                (i,   j+1) in S && push!(clearance, (i,   j+1))
                Gamma[((i,j), t)] = clearance
            end
        end
    end

    L  = Dict(1=>1, 2=>1, 3=>1, 4=>1, 5=>1, 6=>1, 7=>1, 8=>1)
    U  = Dict(1=>2, 2=>4, 3=>2, 4=>4, 5=>3, 6=>4, 7=>2, 8=>3)
    Ac = Dict(c => 0.0 for c in C)
    h  = Dict(c => h_val for c in C)
    r  = Dict(c => rev   for c in C)
    R  = Dict(c => 1     for c in C)

    C_load   = Dict(p => [c for c in C if L[c] == p] for p in P)
    C_unload = Dict(p => [c for c in C if U[c] == p] for p in P)

    W = collect(1:3)
    C_time = Dict(
        (1,1)=>6.0, (1,2)=>5.0, (1,3)=>4.0,
        (2,1)=>6.0, (2,2)=>5.0, (2,3)=>4.0,
        (3,1)=>6.0, (3,2)=>5.0, (3,3)=>4.0,
    )
    C_con = Dict(1=>1.0, 2=>2.0, 3=>4.0)

    M = Float64(length(C))

    return (S=S, E=E, S_no_exit=S_no_exit, P=P, C=C, A=A, Gamma=Gamma,
            L=L, U=U, Ac=Ac, h=h, r=r, R=R,
            C_load=C_load, C_unload=C_unload,
            W=W, C_time=C_time, C_con=C_con,
            C_fuel=c_fuel, C_shift=c_shift, M=M)
end


function build_instance_varied(rev, c_fuel, c_shift, h_val, n_cargo, seed=42)

    unava = Set([(1,10),(2,10),(6,10),(7,10)])
    ramp  = [(3,10),(4,10),(5,10)]

    S = [(i,j) for i in 1:nrows for j in 1:ncols if !((i,j) in unava)]
    E = ramp
    S_no_exit = [s for s in S if !(s in E)]

    P = collect(1:4)
    C = collect(1:n_cargo)

    A     = Tuple{Tuple{Int,Int}, Tuple{Int,Int}}[]
    Gamma = Dict{Tuple{Tuple{Int,Int},Tuple{Int,Int}}, Vector{Tuple{Int,Int}}}()

    for (i,j) in S

        # East — one step forward only
        t = (i, j+1)
        if j+1 <= ncols && t in S
            push!(A, ((i,j), t))
            Gamma[((i,j), t)] = Tuple{Int,Int}[]
        end

        # West — one step backward only
        t = (i, j-1)
        if j-1 >= 1 && t in S
            push!(A, ((i,j), t))
            Gamma[((i,j), t)] = Tuple{Int,Int}[]
        end

        # North-East diagonal
        if i > 1 && j+1 <= ncols
            t = (i-1, j+1)
            if t in S
                push!(A, ((i,j), t))
                clearance = Tuple{Int,Int}[]
                (i-1, j)   in S && push!(clearance, (i-1, j))
                (i,   j+1) in S && push!(clearance, (i,   j+1))
                Gamma[((i,j), t)] = clearance
            end
        end

        # South-East diagonal
        if i < nrows && j+1 <= ncols
            t = (i+1, j+1)
            if t in S
                push!(A, ((i,j), t))
                clearance = Tuple{Int,Int}[]
                (i+1, j)   in S && push!(clearance, (i+1, j))
                (i,   j+1) in S && push!(clearance, (i,   j+1))
                Gamma[((i,j), t)] = clearance
            end
        end
    end

    rng    = MersenneTwister(seed)
    U_vals = [rand(rng, 2:4) for _ in 1:n_cargo]
    L  = Dict(c => 1           for c in C)
    U  = Dict(c => U_vals[c]   for c in C)
    Ac = Dict(c => rand(rng, Exponential(0.5)) for c in C)
    h  = Dict(c => h_val       for c in C)
    r  = Dict(c => rev         for c in C)
    R  = Dict(c => 1           for c in C)

    C_load   = Dict(p => [c for c in C if L[c] == p] for p in P)
    C_unload = Dict(p => [c for c in C if U[c] == p] for p in P)

    W = collect(1:3)
    C_time = Dict(
        (1,1)=>6.0, (1,2)=>5.0, (1,3)=>4.0,
        (2,1)=>6.0, (2,2)=>5.0, (2,3)=>4.0,
        (3,1)=>6.0, (3,2)=>5.0, (3,3)=>4.0,
    )
    C_con = Dict(1=>1.0, 2=>2.0, 3=>4.0)
    M = Float64(n_cargo)

    return (S=S, E=E, S_no_exit=S_no_exit, P=P, C=C, A=A, Gamma=Gamma,
            L=L, U=U, Ac=Ac, h=h, r=r, R=R,
            C_load=C_load, C_unload=C_unload,
            W=W, C_time=C_time, C_con=C_con,
            C_fuel=c_fuel, C_shift=c_shift, M=M)
end

build_instance_varied (generic function with 2 methods)

solve_roro_mip: løser med Highs, variabler: cargo acceptance, placement, shifts, flow, timing, speed/fuel, objekt: maximize revenue - fuel - shift penalty, Constraints c1-c9: placement, capacity, shift detection, flow conservation
Constraints c10-c15: timing, departure times, speed choice, fuel consumption
Returns objective, cargo accepted, shift count, solve time and model


In [3]:
function solve_roro_mip(inst; verbose=false)

    (; S, E, S_no_exit, P, C, A, Gamma, L, U, Ac, h, r, R,
       C_load, C_unload, W, C_time, C_con,
       C_fuel, C_shift, M) = inst

    legs = collect(1:length(P)-1)

    model = Model(HiGHS.Optimizer)
    set_silent(model)

    @variable(model, y[C], Bin)
    @variable(model, x[C, S], Bin)
    @variable(model, delta[C, S, P], Bin)
    @variable(model, d[C, S, P])
    @variable(model, f[C, A, P] >= 0)
    @variable(model, ST[C, P] >= 0)
    @variable(model, Dep[P] >= 0)
    @variable(model, lambda[W, legs], Bin)
    @variable(model, B[legs] >= 0)
    @variable(model, TTS[legs] >= 0)

    # Upper bound on flow
    @constraint(model, [c in C, a in A, p in P],
        f[c,a,p] <= R[c] * y[c]
    )

    @objective(model, Max,
        sum(r[c] * y[c] for c in C)
        - C_fuel  * sum(B[p] for p in legs)
        - C_shift * sum(delta[c,s,p] for c in C, s in S, p in P)
    )

    # (c1)
    @constraint(model, [c in C],
        sum(x[c,s] for s in S) == R[c] * y[c]
    )

    # (c2)
    @constraint(model, [s in S],
        sum(x[c,s] for c in C) <= 1
    )

    # (c3)
    @constraint(model, [c in C, s in S, p in P],
        delta[c,s,p] <= x[c,s]
    )

    # (c4)
    @constraint(model, [c in C],
        sum(d[c,e,L[c]] for e in E) <= R[c] * y[c]
    )

    # (c5)
    @constraint(model, [c in C, s in S_no_exit],
        d[c,s,L[c]] == -x[c,s]
    )

    # (c6)
    @constraint(model, [c in C],
        sum(d[c,e,U[c]] for e in E) >= -R[c] * y[c]
    )

    # (c7)
    @constraint(model, [c in C, s in S_no_exit],
        d[c,s,U[c]] == x[c,s]
    )

    # (c8)
    @constraint(model, [c in C, p in P, s in S],
        sum(f[c,a,p] for a in A if a[2] == s)
        - sum(f[c,a,p] for a in A if a[1] == s)
        == d[c,s,p]
    )

    # (c9)
    for p in P, c in C, dc in C
        c == dc && continue
        U[dc] <= p && continue
        for s in S
            flow_into_s    = [a for a in A if a[2] == s]
            clearance_arcs = [a for a in A if s in Gamma[a]]
            @constraint(model,
                sum(f[c,a,p] for a in flow_into_s)
                + sum(f[c,a,p] for a in clearance_arcs)
                <= M * (1 - x[dc,s] + delta[dc,s,p]))
        end
    end

    # (c10)
    @constraint(model, [c in C],
        ST[c, L[c]] >= Ac[c] - M * (1 - y[c])
    )

    # (c11)
    for p in P, c in union(C_load[p], C_unload[p])
        @constraint(model, Dep[p] >= ST[c,p] + h[c])
    end

    # (c12)
    @constraint(model, [p in legs],
        Dep[p+1] >= Dep[p] + TTS[p]
    )

    # (c13)
    @constraint(model, [p in legs],
        sum(lambda[w,p] for w in W) == 1
    )

    # (c14)
    @constraint(model, [p in legs],
        TTS[p] == sum(C_time[(p,w)] * lambda[w,p] for w in W)
    )

    # (c15)
    @constraint(model, [p in legs],
        B[p] == sum(C_con[w] * lambda[w,p] for w in W)
    )

    t_start = time()
    optimize!(model)
    t_mip   = round(time() - t_start, digits=2)
    status  = termination_status(model)

    if status == OPTIMAL
        obj        = objective_value(model)
        n_accepted = sum(value(y[c]) > 0.5 ? 1 : 0 for c in C)
        n_shifts   = sum(value(delta[c,s,p]) > 0.5 ? 1 : 0
                         for c in C, s in S, p in P)

        if verbose
            println("\nCargo decisions:")
            for c in C
                if value(y[c]) > 0.5
                    slots = [s for s in S if value(x[c,s]) > 0.5]
                    println("  Cargo $c → accepted, slot(s): $slots")
                else
                    println("  Cargo $c → rejected")
                end
            end
            println("\nSailing legs:")
            for p in legs
                w_chosen = findfirst(w -> value(lambda[w,p]) > 0.5, W)
                println("  Leg $p→$(p+1): speed $w_chosen, ",
                        "fuel=$(round(value(B[p]),digits=2)), ",
                        "sail time=$(round(value(TTS[p]),digits=2))")
            end
            println("\nDeparture times:")
            for p in P
                println("  Port $p: $(round(value(Dep[p]),digits=2))")
            end
        end

        return (obj=obj, n_accepted=n_accepted, n_shifts=n_shifts,
                status=status, time=t_mip), model
    else
        return (obj=nothing, n_accepted=0, n_shifts=0,
                status=status, time=t_mip), model
    end
end


function plot_mip_solution(inst, model)

    deck = zeros(Int, nrows, ncols)

    for (i,j) in [(1,10),(2,10),(6,10),(7,10)]
        deck[i,j] = 0
    end
    for (i,j) in inst.S
        deck[i,j] = 1
    end
    for (i,j) in inst.E
        deck[i,j] = 2
    end
    for c in inst.C
        if value(model[:y][c]) > 0.5
            for s in inst.S
                if value(model[:x][c,s]) > 0.5
                    i, j      = s
                    deck[i,j] = inst.U[c] + 2
                end
            end
        end
    end

    println("\nDeck layout:")
    println("  X = unavailable   . = free   R = ramp")
    println("  2 = unloads port 2   3 = unloads port 3   4 = unloads port 4")
    println()
    println("  col→  1    2    3    4    5    6    7    8    9   10")
    println("       ", "-"^50)

    for i in 1:nrows
        print("row $i |")
        for j in 1:ncols
            v = deck[i,j]
            if v == 0
                print("  X  ")
            elseif v == 1
                print("  .  ")
            elseif v == 2
                print("  R  ")
            else
                print("  $(v-2)  ")
            end
        end
        println()
    end
    println()

    return deck
end

plot_mip_solution (generic function with 1 method)

Vi tester her hvordan den vil præstere parametre sat til disse værdier:  revenue = 100 per cargo, Fuel cost = 1.0, Shift penalty = 2.0
og Handling time = 1.0: 

In [4]:
inst       = build_instance(100.0, 1.0, 2.0, 1.0)
res, model = solve_roro_mip(inst, verbose=true)
plot_mip_solution(inst, model)


Cargo decisions:
  Cargo 1 → accepted, slot(s): [(4, 10)]
  Cargo 2 → accepted, slot(s): [(1, 2)]
  Cargo 3 → accepted, slot(s): [(2, 1)]
  Cargo 4 → accepted, slot(s): [(4, 4)]
  Cargo 5 → accepted, slot(s): [(5, 10)]
  Cargo 6 → accepted, slot(s): [(3, 4)]
  Cargo 7 → accepted, slot(s): [(7, 9)]
  Cargo 8 → accepted, slot(s): [(1, 7)]

Sailing legs:
  Leg 1→2: speed 1, fuel=1.0, sail time=6.0
  Leg 2→3: speed 1, fuel=1.0, sail time=6.0
  Leg 3→4: speed 1, fuel=1.0, sail time=6.0

Departure times:
  Port 1: 1.0
  Port 2: 7.0
  Port 3: 13.0
  Port 4: 19.0

Deck layout:
  X = unavailable   . = free   R = ramp
  2 = unloads port 2   3 = unloads port 3   4 = unloads port 4

  col→  1    2    3    4    5    6    7    8    9   10
       --------------------------------------------------
row 1 |  .    4    .    .    .    .    3    .    .    X  
row 2 |  2    .    .    .    .    .    .    .    .    X  
row 3 |  .    .    .    4    .    .    .    .    .    R  
row 4 |  .    .    .    4    .  

7×10 Matrix{Int64}:
 1  6  1  1  1  1  5  1  1  0
 4  1  1  1  1  1  1  1  1  0
 1  1  1  6  1  1  1  1  1  2
 1  1  1  6  1  1  1  1  1  4
 1  1  1  1  1  1  1  1  1  5
 1  1  1  1  1  1  1  1  1  0
 1  1  1  1  1  1  1  1  4  0

Sammenligningen af parametre vha. sensitivitetsanalayse:  

In [5]:
test_cases = [
    #  label                rev    c_fuel  c_shift  h_val
    ("Baseline",           100.0,  1.0,    2.0,     1.0),
    ("Low revenue",         50.0,  1.0,    2.0,     1.0),
    ("High revenue",       150.0,  1.0,    2.0,     1.0),
    ("High fuel cost",     100.0,  3.0,    2.0,     1.0),
    ("High shift penalty", 100.0,  1.0,    5.0,     1.0),
    ("High handling",      100.0,  1.0,    2.0,     2.0),
    ("Very high handling", 100.0,  1.0,    2.0,     3.0),
]

println("\n", "="^70)
println("  RoRo Stowage MIP — Deck A, 8 cargo, 4 ports")
println("="^70)
println(@sprintf("%-22s %8s %8s %8s %8s %8s",
    "Instance", "Obj", "Accepted", "Shifts", "Time(s)", "Status"))
println("-"^70)

for (label, rev, c_fuel, c_shift, h_val) in test_cases
    inst      = build_instance(rev, c_fuel, c_shift, h_val)
    res, _    = solve_roro_mip(inst)
    obj_str   = res.obj === nothing ? "N/A" : string(round(res.obj, digits=2))
    println(@sprintf("%-22s %8s %8d %8d %8.2f %8s",
        label,
        obj_str,
        res.n_accepted,
        res.n_shifts,
        res.time,
        string(res.status)))
end

println("="^70)


  RoRo Stowage MIP — Deck A, 8 cargo, 4 ports
Instance                    Obj Accepted   Shifts  Time(s)   Status
----------------------------------------------------------------------
Baseline                  797.0        8        0     1.40  OPTIMAL
Low revenue               397.0        8        0     1.24  OPTIMAL
High revenue             1197.0        8        0     0.93  OPTIMAL
High fuel cost            791.0        8        0     1.38  OPTIMAL
High shift penalty        797.0        8        0     1.41  OPTIMAL
High handling             797.0        8        0     1.43  OPTIMAL
Very high handling        797.0        8        0     1.40  OPTIMAL


Varierer antal cargo:

In [6]:
using Distributions

function build_instance_varied(rev, c_fuel, c_shift, h_val, n_cargo, seed=42)

    unava = Set([(1,10),(2,10),(6,10),(7,10)])
    ramp  = [(3,10),(4,10),(5,10)]

    S = [(i,j) for i in 1:nrows for j in 1:ncols if !((i,j) in unava)]
    E = ramp
    S_no_exit = [s for s in S if !(s in E)]

    P = collect(1:4)
    C = collect(1:n_cargo)

    A     = Tuple{Tuple{Int,Int}, Tuple{Int,Int}}[]
    Gamma = Dict{Tuple{Tuple{Int,Int},Tuple{Int,Int}}, Vector{Tuple{Int,Int}}}()

    for (i,j) in S
        for j2 in j+1:ncols
            t = (i, j2); !(t in S) && continue
            push!(A, ((i,j), t))
            Gamma[((i,j), t)] = Tuple{Int,Int}[]
        end
        for j2 in 1:j-1
            t = (i, j2); !(t in S) && continue
            push!(A, ((i,j), t))
            Gamma[((i,j), t)] = Tuple{Int,Int}[]
        end
        if i > 1 && j + 1 <= ncols
            t = (i-1, j+1)
            if t in S
                push!(A, ((i,j), t))
                clearance = Tuple{Int,Int}[]
                (i-1, j)   in S && push!(clearance, (i-1, j))
                (i,   j+1) in S && push!(clearance, (i,   j+1))
                Gamma[((i,j), t)] = clearance
            end
        end
        if i < nrows && j + 1 <= ncols
            t = (i+1, j+1)
            if t in S
                push!(A, ((i,j), t))
                clearance = Tuple{Int,Int}[]
                (i+1, j)   in S && push!(clearance, (i+1, j))
                (i,   j+1) in S && push!(clearance, (i,   j+1))
                Gamma[((i,j), t)] = clearance
            end
        end
    end

    rng    = MersenneTwister(seed)
    U_vals = [rand(rng, 2:4) for _ in 1:n_cargo]
    L  = Dict(c => 1           for c in C)
    U  = Dict(c => U_vals[c]   for c in C)
    Ac = Dict(c => rand(rng, Exponential(0.5)) for c in C)
    h  = Dict(c => h_val       for c in C)
    r  = Dict(c => rev         for c in C)
    R  = Dict(c => 1           for c in C)

    C_load   = Dict(p => [c for c in C if L[c] == p] for p in P)
    C_unload = Dict(p => [c for c in C if U[c] == p] for p in P)

    W = collect(1:3)
    C_time = Dict(
        (1,1)=>6.0, (1,2)=>5.0, (1,3)=>4.0,
        (2,1)=>6.0, (2,2)=>5.0, (2,3)=>4.0,
        (3,1)=>6.0, (3,2)=>5.0, (3,3)=>4.0,
    )
    C_con = Dict(1=>1.0, 2=>2.0, 3=>4.0)
    M = Float64(n_cargo)

    return (S=S, E=E, S_no_exit=S_no_exit, P=P, C=C, A=A, Gamma=Gamma,
            L=L, U=U, Ac=Ac, h=h, r=r, R=R,
            C_load=C_load, C_unload=C_unload,
            W=W, C_time=C_time, C_con=C_con,
            C_fuel=c_fuel, C_shift=c_shift, M=M)
end

varied_cases = [
    #  label         rev    c_fuel  c_shift  h_val  n_cargo
    ("8 cargo",   100.0,  1.0,    2.0,     1.0,   8),
    ("15 cargo",   100.0,  1.0,    2.0,     1.0,   12),
    ("High fuel 15",  100.0,  3.0,    2.0,     1.0,  15),
    ("20 cargo",  100.0,  1.0,    2.0,     1.0,  20),
    ("25 cargo",  100.0,  1.0,    2.0,     1.0,  25),
    ("30 cargo",  100.0,  1.0,    2.0,     1.0,  30),
]

println("\n", "="^70)
println("  RoRo Stowage MIP — Deck A, varying cargo, 4 ports")
println("="^70)
println(@sprintf("%-12s %8s %8s %8s %8s %8s",
    "Instance", "Obj", "Accepted", "Shifts", "Time(s)", "Status"))
println("-"^70)

for (label, rev, c_fuel, c_shift, h_val, n_cargo) in varied_cases
    inst    = build_instance_varied(rev, c_fuel, c_shift, h_val, n_cargo)
    res, _  = solve_roro_mip(inst)
    obj_str = res.obj === nothing ? "N/A" : string(round(res.obj, digits=2))
    println(@sprintf("%-12s %8s %8d %8d %8.2f %8s",
        label,
        obj_str,
        res.n_accepted,
        res.n_shifts,
        res.time,
        string(res.status)))
end

println("="^70)


  RoRo Stowage MIP — Deck A, varying cargo, 4 ports
Instance          Obj Accepted   Shifts  Time(s)   Status
----------------------------------------------------------------------
8 cargo         797.0        8        0     3.40  OPTIMAL
15 cargo       1197.0       12        0     1.57  OPTIMAL
High fuel 15   1491.0       15        0    10.11  OPTIMAL
20 cargo       1997.0       20        0    16.58  OPTIMAL
25 cargo       2497.0       25        0    70.45  OPTIMAL
30 cargo       2997.0       30        0   159.58  OPTIMAL


En specifik instance: 

In [7]:
inst      = build_instance_varied(100.0, 1.0, 2.0, 1.0, 25)
res, model = solve_roro_mip(inst, verbose=true)
plot_mip_solution(inst, model)


Cargo decisions:
  Cargo 1 → accepted, slot(s): [(3, 1)]
  Cargo 2 → accepted, slot(s): [(4, 3)]
  Cargo 3 → accepted, slot(s): [(4, 1)]
  Cargo 4 → accepted, slot(s): [(3, 9)]
  Cargo 5 → accepted, slot(s): [(3, 7)]
  Cargo 6 → accepted, slot(s): [(4, 2)]
  Cargo 7 → accepted, slot(s): [(4, 7)]
  Cargo 8 → accepted, slot(s): [(4, 5)]
  Cargo 9 → accepted, slot(s): [(5, 1)]
  Cargo 10 → accepted, slot(s): [(3, 3)]
  Cargo 11 → accepted, slot(s): [(5, 2)]
  Cargo 12 → accepted, slot(s): [(4, 9)]
  Cargo 13 → accepted, slot(s): [(5, 9)]
  Cargo 14 → accepted, slot(s): [(5, 8)]
  Cargo 15 → accepted, slot(s): [(3, 4)]
  Cargo 16 → accepted, slot(s): [(3, 8)]
  Cargo 17 → accepted, slot(s): [(3, 2)]
  Cargo 18 → accepted, slot(s): [(3, 5)]
  Cargo 19 → accepted, slot(s): [(5, 5)]
  Cargo 20 → accepted, slot(s): [(5, 4)]
  Cargo 21 → accepted, slot(s): [(4, 4)]
  Cargo 22 → accepted, slot(s): [(3, 6)]
  Cargo 23 → accepted, slot(s): [(5, 6)]
  Cargo 24 → accepted, slot(s): [(5, 7)]
  Cargo

7×10 Matrix{Int64}:
 1  1  1  1  1  1  1  1  1  0
 1  1  1  1  1  1  1  1  1  0
 6  5  4  6  5  5  4  4  4  2
 5  6  4  5  6  1  4  1  4  2
 6  6  4  4  6  4  5  5  4  2
 1  1  1  1  1  1  1  1  1  0
 1  1  1  1  1  1  1  1  1  0

**1**)

**Vi tester nu heurestikkerne. Først defineres de og herefter testes de:** 

In [8]:
# ============================================================
# CELL 1 — Packages
# ============================================================
import StatsBase: sample, Weights
using Graphs
using SimpleWeightedGraphs
using Distributions
using Random
using JuMP
using HiGHS
using Printf

In [9]:
# ============================================================
# CELL 2 — FROM: src/cargo_generation.jl
# ============================================================
function generate_cargo_type(n; rng=nothing)
    rng !== nothing ? rand(rng, ["car" "truck" "machinery" "container"], n) : rand(["car" "truck" "machinery" "container"], n)
end

function generate_cargo_port(n; num_ports=5, rng=nothing)
    rng !== nothing ? rand(rng, [i for i in 3:num_ports+2], n) : rand([i for i in 3:num_ports+2], n)
end

function generate_arrival_times_exp(n; lambd=0.2, rng=nothing)
    rng !== nothing ? [rand(rng, Exponential(lambd)) for _ in 1:n] : [rand(Exponential(lambd)) for _ in 1:n]
end

function generate_rev(n; mu=1, sig=0.1, rng=nothing)
    rng !== nothing ? [rand(rng, Normal(mu, sig)) for _ in 1:n] : [rand(Normal(mu, sig)) for _ in 1:n]
end

struct Cargo
    type::String
    port::Int32
    arr::Float32
    rev::Float32
end

function genereate_cargo_structs(n; seed=nothing)
    if seed !== nothing
        rng       = MersenneTwister(seed)
        types     = generate_cargo_type(n, rng=rng)
        ports     = generate_cargo_port(n, rng=rng)
        arr_times = generate_arrival_times_exp(n, rng=rng)
        revs      = generate_rev(n, rng=rng)
    else
        types     = generate_cargo_type(n)
        ports     = generate_cargo_port(n)
        arr_times = generate_arrival_times_exp(n)
        revs      = generate_rev(n)
    end
    return [Cargo(types[i], ports[i], arr_times[i], revs[i]) for i in 1:n]
end

genereate_cargo_structs (generic function with 1 method)

In [10]:
# ============================================================
# CELL 3 — FROM: src/deck_representation.jl
# ============================================================
struct Deck
    width::Int32
    length::Int32
    unava::Array{Any}
    ramp::Array{Any}
end

function create_deck(deck)
    w, l = deck.width, deck.length
    outline = ones(Int, w, l)
    for coord in deck.unava
        x, y = coord
        outline[x, y] = 0
    end
    for r in deck.ramp
        x, y = r
        if outline[x, y] != 0
            outline[x, y] = 2
        end
    end
    return outline
end

create_deck (generic function with 1 method)

In [11]:
# ============================================================
# CELL 4 — FROM: src/Decks/DeckA.jl
# ============================================================
w, l = 7, 10
unava = [[1,10],[2,10],[6,10],[7,10]]
ramp  = [[3,10],[4,10],[5,10]]
deckAstruct = Deck(w, l, unava, ramp)
deckAmat    = create_deck(deckAstruct)

7×10 Matrix{Int64}:
 1  1  1  1  1  1  1  1  1  0
 1  1  1  1  1  1  1  1  1  0
 1  1  1  1  1  1  1  1  1  2
 1  1  1  1  1  1  1  1  1  2
 1  1  1  1  1  1  1  1  1  2
 1  1  1  1  1  1  1  1  1  0
 1  1  1  1  1  1  1  1  1  0

In [12]:
# ============================================================
# CELL 5 — FROM: src/tests/min_shifts.jl
# ============================================================
function get_c_loc(deck, port=1)
    portid = port + 2
    id_cargo = []
    for (i, row) in enumerate(eachrow(deck))
        for (j, slot) in enumerate(row)
            if slot == portid
                push!(id_cargo, [i, j])
            end
        end
    end
    return id_cargo
end

function get_ramp_loc(deck)
    ramp_loc = []
    for (i, row) in enumerate(eachrow(deck))
        for (j, slot) in enumerate(row)
            if slot == 2
                push!(ramp_loc, [i, j])
            end
        end
    end
    return ramp_loc
end

function get_graph(deck)
    m, n = size(deck)
    bigM = 100
    g = SimpleWeightedDiGraph(m * n)
    for i in 1:m
        for j in 1:n
            deck[i,j] == 0 && continue
            j == n && continue
            current_node = (i-1)*n + j
            if i > 1 && j < n && deck[i-1,j] > 0 && deck[i-1,j+1] > 0 && deck[i,j+1] > 0
                neighbor_node = (i-2)*n + j + 1
                cost = maximum([sum([(deck[i-1,j] > 3), (deck[i-1,j+1] > 3), (deck[i,j+1] > 3)]), 0.0001])
                add_edge!(g, current_node, neighbor_node, cost)
            end
            if j > 1 && deck[i,j-1] != 0
                neighbor_node = (i-1)*n + (j-1)
                cost = j == n ? bigM : deck[i,j+1] == 0 ? bigM : (deck[i,j-1] > 3 || deck[i,j+1] > 3) ? 1.0 : 0.0001
                add_edge!(g, current_node, neighbor_node, cost)
            end
            if i < m && j < n && deck[i+1,j] > 0 && deck[i+1,j+1] > 0 && deck[i,j+1] > 0
                neighbor_node = i*n + j + 1
                cost = maximum([sum([(deck[i+1,j] > 3), (deck[i+1,j+1] > 3), (deck[i,j+1] > 3)]), 0.0001])
                add_edge!(g, current_node, neighbor_node, cost)
            end
            if j < n && deck[i,j+1] != 0
                neighbor_node = (i-1)*n + (j+1)
                cost = (deck[i,j+1] > 3) ? 1.0 : 0.0001
                add_edge!(g, current_node, neighbor_node, cost)
            end
        end
    end
    return g
end

function shortest_path_like_hansen(deck)
    goal_slots  = get_ramp_loc(deck)
    m, n        = size(deck)
    start_slots = get_c_loc(deck)
    results     = []
    V           = Set{Tuple{Int,Int}}()
    for start_pos in start_slots
        g          = get_graph(deck)
        start_node = (start_pos[1]-1)*n + start_pos[2]
        dsp        = dijkstra_shortest_paths(g, start_node)
        dists      = dsp.dists
        min_dist   = Inf
        best_goal  = nothing
        best_path  = nothing
        for goal_pos in goal_slots
            goal_node = (goal_pos[1]-1)*n + goal_pos[2]
            if dists[goal_node] < min_dist
                min_dist  = dists[goal_node]
                best_goal = goal_pos
                path      = []
                current   = goal_node
                while current != start_node
                    pushfirst!(path, current)
                    current = dsp.parents[current]
                    current == 0 && break
                end
                pushfirst!(path, start_node)
                best_path = path
            end
        end
        if best_path !== nothing
            for i in 1:length(best_path)-1
                u = best_path[i]
                v = best_path[i+1]
                w = Graphs.weights(g)[u, v]
                if isapprox(w, 1.0; atol=1e-9)
                    push!(V, (u, v))
                    add_edge!(g, u, v, 0.0001)
                end
            end
        end
        push!(results, (start_pos, best_goal, min_dist, best_path))
    end
    return results, V
end

function min_shifts_work(deck)
    shortestp = shortest_path_like_hansen(deck)
    vals = [a[3] for a in shortestp[1]]
    isempty(vals) && return 0.0
    return sum(Float64.(vals))
end

function min_shifts(deck)
    min_shifts_work(deck) == Inf && return Inf
    return Float64(length(shortest_path_like_hansen(deck)[2]))
end

function min_shift_all_cargo(deck)
    tot_shifts = min_shifts(deck)
    cur_deck   = copy(deck)
    cur_deck[cur_deck .== 3] .= 1
    cur_deck[cur_deck .> 3]  .-= 1
    for i in 4:maximum(deck)
        tot_shifts += min_shifts(cur_deck)
        cur_deck = copy(cur_deck)
        cur_deck[cur_deck .== 3] .= 1
        cur_deck[cur_deck .> 3]  .-= 1
    end
    return tot_shifts
end

function min_shift_all_cargo_work(deck)
    tot_shifts = min_shifts_work(deck)
    cur_deck   = copy(deck)
    cur_deck[cur_deck .== 3] .= 1
    cur_deck[cur_deck .> 3]  .-= 1
    for i in 4:maximum(deck)
        tot_shifts += min_shifts_work(cur_deck)
        cur_deck = copy(cur_deck)
        cur_deck[cur_deck .== 3] .= 1
        cur_deck[cur_deck .> 3]  .-= 1
    end
    return tot_shifts
end

min_shift_all_cargo_work (generic function with 1 method)

In [13]:
# ============================================================
# CELL 6 — FROM: src/tests/waiting_time.jl
# ============================================================
function wait_time(arr_times, handling_time=4/60, num_operators=3)
    if typeof(arr_times) == Vector{Cargo}
        arr_times = [c.arr for c in arr_times]
    end
    busy_until = zeros(Float64, num_operators)
    for arr_time in arr_times
        earliest_free = argmin(busy_until)
        start_time    = max(arr_time, busy_until[earliest_free])
        busy_until[earliest_free] = start_time + handling_time
    end
    return maximum(busy_until)
end

wait_time (generic function with 3 methods)

In [14]:
# ============================================================
# CELL 7 — FROM: src/tests/obj_func.jl  (fixed version)
# ============================================================
function evaluate_sol(deck, cargo_on; mean_rev_cargo=1500, pcostshift=100, timecost=3000, shift_evaluator="work_tot")
    if cargo_on isa Matrix
        c_on = Cargo[c for row in eachrow(cargo_on) for c in row if c isa Cargo]
    else
        c_on = Cargo[c for c in cargo_on if c isa Cargo]
    end
    isempty(c_on) && return 0.0
    totrev = sum(cargo.rev * mean_rev_cargo for cargo in c_on)
    wcost  = wait_time(c_on) * timecost
    if shift_evaluator == "work"
        shift_cost = min_shifts_work(deck) * pcostshift
    elseif shift_evaluator == "shifts"
        shift_cost = min_shifts(deck) * pcostshift
    elseif shift_evaluator == "shifts_tot"
        shift_cost = min_shift_all_cargo(deck) * pcostshift
    elseif shift_evaluator == "work_tot"
        shift_cost = min_shift_all_cargo_work(deck) * pcostshift
    end
    return totrev - wcost - shift_cost
end

function sol_details(deck, cargo_on; mean_rev_cargo=1500, pcostshift=100, timecost=3000)
    c_on       = Cargo[c for row in eachrow(cargo_on) for c in row if c isa Cargo]
    totrev     = sum(cargo.rev * mean_rev_cargo for cargo in c_on)
    wcost      = wait_time(c_on) * timecost
    shift_cost = min_shifts(deck) * pcostshift
    return (totrev, wcost, shift_cost)
end

sol_details (generic function with 1 method)

In [15]:
# ============================================================
# CELL 8 — FROM: src/Heuristics/priorityQ.jl
# ============================================================
function pri_rules1(deck, cargo)
    cargoDict = Dict()
    scoreList = []
    h, w      = size(deck)
    for (i, c) in enumerate(cargo)
        cargoDict["c$i"] = c
    end
    for (k, v) in cargoDict
        push!(scoreList, [k, 1/v.port + v.arr])
    end
    scoreList = sort(scoreList, by=x -> x[2])
    cargo_on  = Matrix{Any}(nothing, h, w)
    for score in scoreList
        (id, _) = score
        cport   = cargoDict[id].port
        placed  = false
        for (j, col) in enumerate(eachcol(deck))
            placed && continue
            for (i, slot) in enumerate(col)
                placed && continue
                if slot == 1
                    deck[i,j]     = cport
                    cargo_on[i,j] = cargoDict[id]
                    placed        = true
                end
            end
        end
    end
    return deck, cargo_on
end

function pri_rules2(deck, cargo; a=0.5)
    cargoDict = Dict()
    scoreList = []
    deck      = copy(deck)
    cargo     = copy(cargo)
    h, w      = size(deck)
    for (i, c) in enumerate(cargo)
        cargoDict["c$i"] = c
    end
    for (k, v) in cargoDict
        push!(scoreList, [k, a*(1/v.port) + (1-a)*v.arr])
    end
    scoreList = sort(scoreList, by=x -> x[2])
    cargo_on  = Matrix{Any}(nothing, h, w)
    for score in scoreList
        (id, _) = score
        cport   = cargoDict[id].port
        carg    = cargoDict[id]
        placed  = false
        for (j, col) in enumerate(eachcol(deck))
            placed && continue
            for (i, slot) in enumerate(col)
                placed && continue
                if slot == 1
                    deck[i,j]     = cport
                    cargo_on[i,j] = carg
                    placed        = true
                end
            end
        end
    end
    return deck, cargo_on
end

pri_rules2 (generic function with 1 method)

In [16]:
# ============================================================
# CELL 9 — FROM: src/Heuristics/load_random.jl
# ============================================================
function load_random(deck, cargo)
    deck     = copy(deck)
    cargo    = copy(cargo)
    h, w     = size(deck)
    cargo_on = Matrix{Any}(nothing, h, w)
    while !isempty(cargo) && 1 in deck
        i = rand(1:h)
        j = rand(1:w)
        if deck[i,j] == 1
            current_cargo    = popfirst!(cargo)
            deck[i,j]        = current_cargo.port
            cargo_on[i,j]    = current_cargo
        end
    end
    return deck, cargo_on
end

load_random (generic function with 1 method)

In [17]:
# ============================================================
# CELL 10 — FROM: src/Heuristics/alns.jl
# ============================================================
function get_neighbors(deck, i, j)
    neighs = []
    h, w   = size(deck)
    i != 1 && push!(neighs, deck[i-1,j])
    j != 1 && push!(neighs, deck[i,j-1])
    i != h && push!(neighs, deck[i+1,j])
    j != w && push!(neighs, deck[i,j+1])
    return neighs
end

function destroy_neighbor(deck, cargo_on; xi=0.2)
    deck     = copy(deck)
    cargo_on = copy(cargo_on)
    h, w     = size(deck)
    L        = []
    Lloc     = []
    n_cargo  = count(x -> x > 2, deck)
    for (i, row) in enumerate(eachrow(deck))
        for (j, slot) in enumerate(row)
            if slot > 2 && !(slot in get_neighbors(deck, i, j))
                push!(Lloc, [i,j])
            end
        end
    end
    for (count, loc) in enumerate(shuffle!(Lloc))
        count > xi * n_cargo && return deck, L, cargo_on
        push!(L, cargo_on[loc[1], loc[2]])
        cargo_on[loc[1], loc[2]] = nothing
        deck[loc[1], loc[2]]     = 1
    end
    return deck, L, cargo_on
end

function destroy_area(deck, cargo_on; xi=0.1)
    deck            = copy(deck)
    cargo_on        = copy(cargo_on)
    h, w            = size(deck)
    n_cargo         = count(!isnothing, cargo_on)
    cargo_positions = findall(!isnothing, cargo_on)
    target          = ceil(Int, n_cargo * xi)
    cargo2place     = Any[]
    start_loc       = rand(cargo_positions)
    start_i, start_j = start_loc[1], start_loc[2]
    area_height     = 0
    area_width      = 0
    while length(cargo2place) < target && area_height <= h && area_width <= w
        for i in start_i:(start_i + area_height)
            for j in start_j:(start_j + area_width)
                (i < 1 || i > h || j < 1 || j > w) && continue
                if deck[i,j] > 2 && !isnothing(cargo_on[i,j])
                    push!(cargo2place, cargo_on[i,j])
                    deck[i,j]     = 1
                    cargo_on[i,j] = nothing
                end
                length(cargo2place) >= target && break
            end
            length(cargo2place) >= target && break
        end
        rand() > 0.5 ? area_height += 1 : area_width += 1
    end
    return deck, cargo2place, cargo_on
end

function destroy_random(deck, cargo_on; xi=0.2)
    deck     = copy(deck)
    cargo_on = copy(cargo_on)
    h, w     = size(deck)
    n_cargo  = count(x -> x > 2, deck)
    L        = []
    while length(L) < xi * n_cargo
        i = rand(1:h)
        j = rand(1:w)
        if deck[i,j] > 2
            push!(L, cargo_on[i,j])
            deck[i,j]     = 1
            cargo_on[i,j] = nothing
        end
    end
    return deck, L, cargo_on
end

function destroy_port(deck, cargo_on; xi=0.2)
    ports    = unique(deck[deck .> 2])
    port2rem = rand(ports)
    deck     = copy(deck)
    cargo_on = copy(cargo_on)
    h, w     = size(deck)
    n_cargo  = count(x -> x == port2rem, deck)
    L        = []
    while length(L) < xi * n_cargo
        i = rand(1:h)
        j = rand(1:w)
        if deck[i,j] == port2rem
            push!(L, cargo_on[i,j])
            deck[i,j]     = 1
            cargo_on[i,j] = nothing
        end
    end
    return deck, L, cargo_on
end

function destroy_shifting_cost(deck, cargo_on; xi=0.2)
    deck     = copy(deck)
    cargo_on = copy(cargo_on)
    _, V     = shortest_path_like_hansen(deck)
    h, w     = size(deck)
    n_cargo  = count(x -> x > 2, deck)
    if isempty(V)
        cargo_positions     = [[i,j] for i in 1:h for j in 1:w if deck[i,j] > 2]
        n_selected_blockers = max(1, round(Int, xi * n_cargo / 2))
        selected_blockers   = shuffle!(cargo_positions)[1:min(n_selected_blockers, length(cargo_positions))]
    else
        blockers            = [[div(num[2], w)+1, num[2]%w+1] for num in V]
        n_selected_blockers = max(1, round(Int, xi/2 * length(blockers)))
        selected_blockers   = shuffle!(blockers)[1:min(n_selected_blockers, length(blockers))]
    end
    rad  = length(selected_blockers)
    Lloc = []
    L    = []
    for slot in selected_blockers
        push!(Lloc, slot)
        i, j = slot
        for _ in 1:rad
            i != 1 && (push!(Lloc, [i-1,j]); j != 1 && push!(Lloc, [i-1,j-1]); j != w && push!(Lloc, [i-1,j+1]))
            i != h && (push!(Lloc, [i+1,j]); j != 1 && push!(Lloc, [i+1,j-1]); j != w && push!(Lloc, [i+1,j+1]))
            j != 1 && push!(Lloc, [i,j-1])
            j != w && push!(Lloc, [i,j+1])
        end
    end
    for loc in Lloc
        i, j = loc
        if deck[i,j] > 2
            push!(L, cargo_on[i,j])
            deck[i,j]     = 1
            cargo_on[i,j] = nothing
        end
    end
    return deck, L, cargo_on
end

destroy_shifting_cost (generic function with 1 method)

In [18]:
# ============================================================
# CELL 11 — FROM: src/Heuristics/alns.jl (repair operators)
# ============================================================
function repair_greedy(deck, cargo2place, cargo_on)
    cargoDict = Dict()
    scoreList = []
    deck      = copy(deck)
    cargo     = copy(cargo2place)
    h, w      = size(deck)
    for (i, c) in enumerate(cargo)
        cargoDict["c$i"] = c
    end
    for (k, v) in cargoDict
        push!(scoreList, [k, 1/v.port + v.arr])
    end
    scoreList = sort(scoreList, by=x -> x[2])
    for score in scoreList
        (id, _) = score
        cport   = cargoDict[id].port
        carg    = cargoDict[id]
        placed  = false
        for (j, col) in enumerate(eachcol(deck))
            placed && continue
            for (i, slot) in enumerate(col)
                placed && continue
                if slot == 1
                    deck[i,j]     = cport
                    cargo_on[i,j] = carg
                    placed        = true
                end
            end
        end
    end
    return deck, cargo_on
end

function repair_neighbor_rand(deck, cargo2place, cargo_on)
    deck        = copy(deck)
    cargo_on    = copy(cargo_on)
    h, w        = size(deck)
    locs        = shuffle!([[i,j] for i in 1:h for j in 1:w if deck[i,j] == 1])
    for loc in locs
        i, j = loc[1], loc[2]
        if !isempty(cargo2place)
            placed = false
            for cargo in shuffle!(copy(cargo2place))
                if !placed && cargo.port in get_neighbors(deck, i, j)
                    cargo_on[i,j] = cargo
                    deck[i,j]     = cargo.port
                    filter!(x -> x !== cargo, cargo2place)
                    placed = true
                end
            end
        end
    end
    if !isempty(cargo2place)
        for i in 1:size(deck,1), j in 1:size(deck,2)
            if deck[i,j] == 1 && !isempty(cargo2place)
                c             = popfirst!(cargo2place)
                cargo_on[i,j] = c
                deck[i,j]     = c.port
            end
        end
    end
    return deck, cargo_on
end

function repair_placement(deck, cargo2place, cargo_on)
    deck       = copy(deck)
    g_deck     = copy(deck)
    g_deck[g_deck .> 2] .= 1
    m, n       = size(deck)
    free       = [(c[1], c[2]) for c in findall(x -> x == 1, g_deck)]
    g          = get_graph(g_deck)
    goal_slots = get_ramp_loc(g_deck)
    goal_pos   = [(loc[1]-1)*n + loc[2] for loc in goal_slots]
    furthest_locs = []
    for loc in free
        pos = (loc[1]-1)*n + loc[2]
        dsp = dijkstra_shortest_paths(g, pos)
        dis = minimum([dsp.dists[i] for i in goal_pos])
        push!(furthest_locs, [loc, dis])
    end
    sort!(furthest_locs, by=x -> x[2], rev=true)
    sort!(cargo2place, by=x -> x.port)
    for loc in furthest_locs
        isempty(cargo2place) && return deck, cargo_on
        i, j = loc[1][1], loc[1][2]
        if deck[i,j] == 1 && isnothing(cargo_on[i,j])
            c             = popfirst!(cargo2place)
            deck[i,j]     = c.port
            cargo_on[i,j] = c
        end
    end
    return deck, cargo_on
end

function repair_random(deck, cargo2place, cargo_on)
    isempty(cargo2place) && return deck, cargo_on
    deck        = copy(deck)
    h, w        = size(deck)
    locs        = shuffle!([[i,j] for i in 1:h for j in 1:w if deck[i,j] == 1])
    n_to_place  = min(length(locs), length(cargo2place))
    for i in 1:n_to_place
        loc           = locs[i]
        c2p           = cargo2place[i]
        deck[loc[1], loc[2]]     = c2p.port
        cargo_on[loc[1], loc[2]] = c2p
    end
    deleteat!(cargo2place, 1:n_to_place)
    return deck, cargo_on
end

repair_random (generic function with 1 method)

In [19]:
# ============================================================
# CELL 12 — FROM: src/tests/run_alns.jl
# ============================================================
function alns_hansen(deck, cargo;
        destroy_ops = [destroy_neighbor, destroy_area, destroy_port, destroy_random, destroy_shifting_cost],
        repair_ops  = [repair_greedy, repair_neighbor_rand, repair_placement, repair_random],
        init        = load_random,
        iterations  = 10000,
        time_lim    = 10000,
        segment     = 100,
        rho         = 0.1)

    t1               = time()
    sig1, sig2, sig3 = 33, 9, 3
    nd, nr           = length(destroy_ops), length(repair_ops)
    w_d, w_r         = ones(nd), ones(nr)
    score_d, score_r = zeros(nd), zeros(nr)
    use_d, use_r     = zeros(nd), zeros(nr)

    best_deck, best_cargo = init(deck, cargo)
    best_val              = evaluate_sol(best_deck, best_cargo)
    current_deck          = copy(best_deck)
    current_cargo         = copy(best_cargo)
    current_val           = best_val
    history               = []

    for it in 1:iterations
        if time() - t1 > time_lim
            println("ran $it iterations and $(time()-t1) seconds")
            return best_deck, history
        end
        d = sample(1:nd, Weights(w_d))
        r = sample(1:nr, Weights(w_r))
        use_d[d] += 1; use_r[r] += 1

        destroyed_deck, cargo2place, destroyed_cargo = destroy_ops[d](current_deck, current_cargo)
        new_deck, new_cargo = repair_ops[r](destroyed_deck, cargo2place, destroyed_cargo)
        new_val = evaluate_sol(new_deck, new_cargo)

        if new_val > best_val
            best_deck, best_cargo, best_val = new_deck, new_cargo, new_val
            current_deck, current_cargo, current_val = new_deck, new_cargo, new_val
            score_d[d] += sig1; score_r[r] += sig1
        elseif new_val > current_val
            current_deck, current_cargo, current_val = new_deck, new_cargo, new_val
            score_d[d] += sig2; score_r[r] += sig2
        elseif rand() < 0.1
            current_deck, current_cargo, current_val = new_deck, new_cargo, new_val
            score_d[d] += sig3; score_r[r] += sig3
        end

        push!(history, best_val)

        if it % segment == 0
            for i in 1:nd
                use_d[i] > 0 && (w_d[i] = (1-rho)*w_d[i] + rho*(score_d[i]/use_d[i]))
            end
            for i in 1:nr
                use_r[i] > 0 && (w_r[i] = (1-rho)*w_r[i] + rho*(score_r[i]/use_r[i]))
            end
            score_d .= 0; score_r .= 0; use_d .= 0; use_r .= 0
        end
    end
    println("ran $iterations iterations and $(time()-t1) seconds")
    return best_deck, history
end

alns_hansen (generic function with 1 method)

Kører nu et eksempel med 8 stykker cargo:

In [20]:
# ============================================================
# CELL 13 — Run heuristic
# ============================================================
carg = genereate_cargo_structs(8, seed=42)
best_deck, history = alns_hansen(deckAmat, carg, time_lim=30)
println("Heuristic objective: ", last(history))

ran 10000 iterations and 3.3758468627929688 seconds
Heuristic objective: 10474.82398724556


Vi vil nu sammenligne vores objektfunktion og heurestik, hvis har samme deck (A) og samme antal stykker cargo: 

In [21]:
# ============================================================
# CELL 14 — Comparison: MIP vs Heuristic
# Same cargo, same deck, compared on shifts and cargo placed
# ============================================================

function mip_to_heuristic_cargo(inst)
    cargo = Cargo[]
    for c in inst.C
        port = Int32(inst.U[c] + 2)
        arr  = Float32(inst.Ac[c])
        rev  = Float32(inst.r[c])
        push!(cargo, Cargo("car", port, arr, rev))
    end
    return cargo
end

test_cases = [
    ("Baseline",           100.0,  1.0,  2.0,  1.0),
    ("Low revenue",         50.0,  1.0,  2.0,  1.0),
    ("High revenue",       150.0,  1.0,  2.0,  1.0),
    ("High fuel cost",     100.0,  3.0,  2.0,  1.0),
    ("High shift penalty", 100.0,  1.0,  5.0,  1.0),
    ("High handling",      100.0,  1.0,  2.0,  2.0),
    ("Very high handling", 100.0,  1.0,  2.0,  3.0),
]

println("\n", "="^85)
println("  MIP vs Heuristic — Deck A, 8 cargo, 4 ports")
println("  Comparison on: cargo accepted and shift quality")
println("="^85)
println(@sprintf("%-22s %8s %8s %8s %8s %8s %8s",
    "Instance", "MIP Obj", "MIP Acc", "MIP Sh", "H Acc", "H Sh", "H Time(s)"))
println("-"^85)

for (label, rev, c_fuel, c_shift, h_val) in test_cases

    # 1. Solve MIP
    inst    = build_instance(rev, c_fuel, c_shift, h_val)
    res, _  = solve_roro_mip(inst)

    # 2. Convert cargo and run heuristic on same cargo
    cargo_h = mip_to_heuristic_cargo(inst)
    t_start = time()
    best_deck_h, history_h = alns_hansen(deckAmat, copy(cargo_h), time_lim=30)
    t_heur  = round(time() - t_start, digits=2)

    # 3. Evaluate heuristic shift count
    h_accepted = count(x -> x > 2, best_deck_h)
    h_shifts   = round(min_shift_all_cargo_work(best_deck_h), digits=4)

    println(@sprintf("%-22s %8.2f %8d %8d %8d %8.4f %8.2f",
        label,
        res.obj === nothing ? 0.0 : res.obj,
        res.n_accepted,
        res.n_shifts,
        h_accepted,
        h_shifts,
        t_heur))
end

println("="^85)


  MIP vs Heuristic — Deck A, 8 cargo, 4 ports
  Comparison on: cargo accepted and shift quality
Instance                MIP Obj  MIP Acc   MIP Sh    H Acc     H Sh H Time(s)
-------------------------------------------------------------------------------------
ran 10000 iterations and 1.8554298877716064 seconds
Baseline                 797.00        8        0        8   0.0028     1.86
ran 10000 iterations and 1.8057641983032227 seconds
Low revenue              397.00        8        0        8   0.0023     1.81
ran 10000 iterations and 1.7448301315307617 seconds
High revenue            1197.00        8        0        8   0.0017     1.75
ran 10000 iterations and 1.7496089935302734 seconds
High fuel cost           791.00        8        0        8   0.0019     1.75
ran 10000 iterations and 1.805053949356079 seconds
High shift penalty       797.00        8        0        8   0.0025     1.81
ran 10000 iterations and 1.709671974182129 seconds
High handling            797.00        8    

Hvordan placerer de cargo? Det undersøger vi her: 

In [22]:
# ============================================================
# CELL 15 — Visualize MIP vs Heuristic placement
# ============================================================

function compare_placements(rev, c_fuel, c_shift, h_val, n_cargo; seed=42)

    # 1. Build and solve MIP
    inst       = build_instance_varied(rev, c_fuel, c_shift, h_val, n_cargo, seed)
    res, model = solve_roro_mip(inst)

    # 2. Get MIP deck layout
    mip_deck = zeros(Int, nrows, ncols)
    for (i,j) in [(1,10),(2,10),(6,10),(7,10)]; mip_deck[i,j] = 0; end
    for (i,j) in inst.S; mip_deck[i,j] = 1; end
    for (i,j) in inst.E; mip_deck[i,j] = 2; end
    for c in inst.C
        if value(model[:y][c]) > 0.5
            for s in inst.S
                if value(model[:x][c,s]) > 0.5
                    i, j = s
                    mip_deck[i,j] = inst.U[c] + 2
                end
            end
        end
    end

    # 3. Run heuristic on same cargo
    cargo_h = mip_to_heuristic_cargo(inst)
    best_deck_h, history_h = alns_hansen(deckAmat, copy(cargo_h), time_lim=30)

    # 4. Print side by side
    println("\n", "="^60)
    println("  MIP vs Heuristic — $n_cargo cargo, Deck A")
    println("  MIP obj: $(round(res.obj, digits=2))   MIP shifts: $(res.n_shifts)")
    println("  Heur obj: $(round(last(history_h), digits=2))   Heur shifts: $(round(min_shift_all_cargo_work(best_deck_h), digits=4))")
    println("="^60)
    println("encoding: . = free, R = ramp, X = unavailable")
    println("          2 = port2, 3 = port3, 4 = port4")
    println()
    println(@sprintf("%-30s %s", "MIP placement:", "Heuristic placement:"))
    println(@sprintf("%-30s %s", "col→ 1  2  3  4  5  6  7  8  9 10", "col→ 1  2  3  4  5  6  7  8  9 10"))
    println(@sprintf("%-30s %s", "     " * "-"^25, "     " * "-"^25))

    for i in 1:nrows
        mip_row  = ""
        heur_row = ""
        for j in 1:ncols
            # MIP
            v = mip_deck[i,j]
            if v == 0;      mip_row *= "  X"
            elseif v == 1;  mip_row *= "  ."
            elseif v == 2;  mip_row *= "  R"
            else;           mip_row *= "  $(v-2)"
            end
            # Heuristic
            v2 = best_deck_h[i,j]
            if v2 == 0;     heur_row *= "  X"
            elseif v2 == 1; heur_row *= "  ."
            elseif v2 == 2; heur_row *= "  R"
            else;           heur_row *= "  $(v2-2)"
            end
        end
        println(@sprintf("row %d |%-25s    row %d |%s", i, mip_row, i, heur_row))
    end
    println()
end

# Run comparison for different cargo counts
compare_placements(100.0, 1.0, 2.0, 1.0, 8)
compare_placements(100.0, 1.0, 2.0, 1.0, 15)
compare_placements(100.0, 1.0, 2.0, 1.0, 20)

ran 10000 iterations and 2.1308770179748535 seconds

  MIP vs Heuristic — 8 cargo, Deck A
  MIP obj: 797.0   MIP shifts: 0
  Heur obj: 1.19854664e6   Heur shifts: 0.002
encoding: . = free, R = ramp, X = unavailable
          2 = port2, 3 = port3, 4 = port4

MIP placement:                 Heuristic placement:
col→ 1  2  3  4  5  6  7  8  9 10 col→ 1  2  3  4  5  6  7  8  9 10
     -------------------------      -------------------------
row 1 |  .  .  .  .  .  .  .  .  .  X    row 1 |  .  .  .  .  .  .  .  .  .  X
row 2 |  .  .  .  .  .  .  2  .  .  X    row 2 |  .  .  .  .  3  .  .  .  .  X
row 3 |  4  .  .  4  2  .  .  .  .  R    row 3 |  .  .  .  .  .  .  .  4  4  R
row 4 |  .  3  .  2  .  .  .  .  .  R    row 4 |  .  .  .  .  .  2  2  .  4  R
row 5 |  4  .  .  .  .  .  .  .  .  R    row 5 |  .  .  .  .  .  .  2  .  2  R
row 6 |  .  .  .  2  .  .  .  .  .  X    row 6 |  .  .  .  .  .  .  .  .  .  X
row 7 |  .  .  .  .  .  .  .  .  .  X    row 7 |  .  .  .  .  .  .  .  .  .  X

ran 10

**3)**

Vi bygger nu en objektfunktion, som matcher heurestikkerne: 

In [23]:
# ============================================================
# SIMPLIFIED MIP — Objective aligned with heuristic
# Revenue × 1500 - WaitTime × 3000 - Shifts × 100
# Removes fuel/speed/timing — WaitTime is a fixed parameter
# ============================================================

function solve_roro_mip_heuristic_obj(inst, cargo_h; verbose=false)

    (; S, E, S_no_exit, P, C, A, Gamma, L, U, R, M) = inst

    # Aligned cost coefficients
    mean_rev_cargo = 1500.0
    timecost       = 3000.0
    pcostshift     = 100.0

    # Compute WaitTime as fixed parameter — exactly as heuristic does
    wait_cost = wait_time(cargo_h) * timecost

    model = Model(HiGHS.Optimizer)
    set_silent(model)

    @variable(model, y[C], Bin)
    @variable(model, x[C, S], Bin)
    @variable(model, delta[C, S, P], Bin)
    @variable(model, d[C, S, P])
    @variable(model, f[C, A, P] >= 0)

    # Upper bound on flow
    @constraint(model, [c in C, a in A, p in P],
        f[c,a,p] <= R[c] * y[c]
    )

    # Objective — exactly matches heuristic
    @objective(model, Max,
        sum(mean_rev_cargo * inst.r[c] * y[c] for c in C)
        - wait_cost
        - pcostshift * sum(delta[c,s,p] for c in C, s in S, p in P)
    )

    # (c1)
    @constraint(model, [c in C],
        sum(x[c,s] for s in S) == R[c] * y[c]
    )

    # (c2)
    @constraint(model, [s in S],
        sum(x[c,s] for c in C) <= 1
    )

    # (c3)
    @constraint(model, [c in C, s in S, p in P],
        delta[c,s,p] <= x[c,s]
    )

    # (c4)
    @constraint(model, [c in C],
        sum(d[c,e,L[c]] for e in E) <= R[c] * y[c]
    )

    # (c5)
    @constraint(model, [c in C, s in S_no_exit],
        d[c,s,L[c]] == -x[c,s]
    )

    # (c6)
    @constraint(model, [c in C],
        sum(d[c,e,U[c]] for e in E) >= -R[c] * y[c]
    )

    # (c7)
    @constraint(model, [c in C, s in S_no_exit],
        d[c,s,U[c]] == x[c,s]
    )

    # (c8)
    @constraint(model, [c in C, p in P, s in S],
        sum(f[c,a,p] for a in A if a[2] == s)
        - sum(f[c,a,p] for a in A if a[1] == s)
        == d[c,s,p]
    )

    # (c9)
    for p in P, c in C, dc in C
        c == dc && continue
        U[dc] <= p && continue
        for s in S
            flow_into_s    = [a for a in A if a[2] == s]
            clearance_arcs = [a for a in A if s in Gamma[a]]
            @constraint(model,
                sum(f[c,a,p] for a in flow_into_s)
                + sum(f[c,a,p] for a in clearance_arcs)
                <= M * (1 - x[dc,s] + delta[dc,s,p]))
        end
    end

    t_start = time()
    optimize!(model)
    t_mip   = round(time() - t_start, digits=2)
    status  = termination_status(model)

    if status == OPTIMAL
        obj        = objective_value(model)
        n_accepted = sum(value(y[c]) > 0.5 ? 1 : 0 for c in C)
        n_shifts   = sum(value(delta[c,s,p]) > 0.5 ? 1 : 0
                         for c in C, s in S, p in P)

        if verbose
            println("\nCargo decisions:")
            for c in C
                if value(y[c]) > 0.5
                    slots = [s for s in S if value(x[c,s]) > 0.5]
                    println("  Cargo $c → accepted, slot(s): $slots")
                else
                    println("  Cargo $c → rejected")
                end
            end
        end

        return (obj=obj, n_accepted=n_accepted, n_shifts=n_shifts,
                wait_cost=wait_cost, status=status, time=t_mip), model
    else
        return (obj=nothing, n_accepted=0, n_shifts=0,
                wait_cost=wait_cost, status=status, time=t_mip), model
    end
end

# ============================================================
# COMPARISON TABLE — aligned objective
# ============================================================

function mip_to_heuristic_cargo(inst)
    cargo = Cargo[]
    for c in inst.C
        port = Int32(inst.U[c] + 2)
        arr  = Float32(inst.Ac[c])
        rev  = Float32(inst.r[c])
        push!(cargo, Cargo("car", port, arr, rev))
    end
    return cargo
end

test_cases = [
    ("Baseline",           100.0,  1.0,  2.0,  1.0),
    ("Low revenue",         50.0,  1.0,  2.0,  1.0),
    ("High revenue",       150.0,  1.0,  2.0,  1.0),
    ("High fuel cost",     100.0,  3.0,  2.0,  1.0),
    ("High shift penalty", 100.0,  1.0,  5.0,  1.0),
    ("High handling",      100.0,  1.0,  2.0,  2.0),
    ("Very high handling", 100.0,  1.0,  2.0,  3.0),
]

println("\n", "="^90)
println("  MIP (aligned obj) vs Heuristic — Deck A, 8 cargo, 4 ports")
println("  Objective: revenue×1500 - WaitTime×3000 - Shifts×100")
println("="^90)
println(@sprintf("%-22s %12s %12s %10s %8s %8s",
    "Instance", "MIP Obj", "Heur Obj", "Gap (%)", "MIP Sh", "H Sh"))
println("-"^90)

for (label, rev, c_fuel, c_shift, h_val) in test_cases

    # 1. Build instance
    inst    = build_instance(rev, c_fuel, c_shift, h_val)

    # 2. Generate matching cargo for heuristic
    cargo_h = mip_to_heuristic_cargo(inst)

    # 3. Solve simplified MIP with heuristic objective
    res, _  = solve_roro_mip_heuristic_obj(inst, cargo_h)

    # 4. Run heuristic on same cargo
    best_deck_h, history_h = alns_hansen(deckAmat, copy(cargo_h), time_lim=30)
    heur_obj   = last(history_h)
    heur_shifts = round(min_shift_all_cargo_work(best_deck_h), digits=4)

    # 5. Gap
    gap = res.obj !== nothing ? round((res.obj - heur_obj) / res.obj * 100, digits=2) : NaN

    println(@sprintf("%-22s %12.2f %12.2f %9.2f%% %8d %8.4f",
        label,
        res.obj === nothing ? 0.0 : res.obj,
        heur_obj,
        gap,
        res.n_shifts,
        heur_shifts))
end

println("="^90)


  MIP (aligned obj) vs Heuristic — Deck A, 8 cargo, 4 ports
  Objective: revenue×1500 - WaitTime×3000 - Shifts×100
Instance                    MIP Obj     Heur Obj    Gap (%)   MIP Sh     H Sh
------------------------------------------------------------------------------------------
ran 10000 iterations and 2.0648181438446045 seconds
Baseline                 1199400.00   1199399.79      0.00%        0   0.0021
ran 10000 iterations and 2.038879156112671 seconds
Low revenue               599400.00    599399.84      0.00%        0   0.0016
ran 10000 iterations and 2.0705041885375977 seconds
High revenue             1799400.00   1799399.85      0.00%        0   0.0015
ran 10000 iterations and 2.075016975402832 seconds
High fuel cost           1199400.00   1199399.81      0.00%        0   0.0019
ran 10000 iterations and 2.057173013687134 seconds
High shift penalty       1199400.00   1199399.48      0.00%        0   0.0052
ran 10000 iterations and 2.0622310638427734 seconds
High handling   

Sammenligner nu med mere cargo: 

In [24]:
test_cases = [
    ("10 cargo",   1.0,  1.0,  2.0,  1.0,  10),
    ("15 cargo",  1.0,  1.0,  2.0,  1.0, 15),
    ("20 cargo",  1.0,  1.0,  2.0,  1.0, 20),
    ("25 cargo",  1.0,  1.0,  2.0,  1.0, 25),
    ("30 cargo",  1.0,  1.0,  2.0,  1.0, 30),
]

println("\n", "="^90)
println("  MIP (aligned obj) vs Heuristic — Deck A, varying cargo, 4 ports")
println("  Objective: revenue x 1500 - WaitTime x 3000 - Shifts x 100")
println("="^90)
println(@sprintf("%-12s %12s %12s %10s %8s %8s",
    "Instance", "MIP Obj", "Heur Obj", "Gap (%)", "MIP Sh", "H Sh"))
println("-"^90)

for (label, rev, c_fuel, c_shift, h_val, n_cargo) in test_cases
    inst    = build_instance_varied(rev, c_fuel, c_shift, h_val, n_cargo)
    cargo_h = mip_to_heuristic_cargo(inst)
    res, _  = solve_roro_mip_heuristic_obj(inst, cargo_h)
    best_deck_h, history_h = alns_hansen(deckAmat, copy(cargo_h), time_lim=30)
    heur_obj    = last(history_h)
    heur_shifts = round(min_shift_all_cargo_work(best_deck_h), digits=4)
    gap = res.obj !== nothing ? round((res.obj - heur_obj) / res.obj * 100, digits=2) : NaN
    println(@sprintf("%-12s %12.2f %12.2f %9.2f%% %8d %8.4f",
        label,
        res.obj === nothing ? 0.0 : res.obj,
        heur_obj,
        gap,
        res.n_shifts,
        heur_shifts))
end

println("="^90)


  MIP (aligned obj) vs Heuristic — Deck A, varying cargo, 4 ports
  Objective: revenue x 1500 - WaitTime x 3000 - Shifts x 100
Instance          MIP Obj     Heur Obj    Gap (%)   MIP Sh     H Sh
------------------------------------------------------------------------------------------
ran 10000 iterations and 3.3196849822998047 seconds
10 cargo         13280.43     13546.45     -2.00%        0   0.0039
ran 10000 iterations and 4.552733898162842 seconds
15 cargo         19323.24     19322.84      0.00%        0   0.0040
ran 10000 iterations and 5.918686866760254 seconds
20 cargo         26823.24     26822.61      0.00%        0   0.0063
ran 10000 iterations and 7.083899021148682 seconds
25 cargo         33710.99     33710.03      0.00%        0   0.0096
ran 10000 iterations and 8.632827997207642 seconds
30 cargo         41010.99     41209.62     -0.48%        0   0.0137
